In [ ]:
import numpy as np
import json
import cvxpy as cp
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

from src.problems.utils import sample_data_for_group
from src.problems.problems import (
    _compute_consumer_optimal_solution_cvar,
    _compute_consumer_optimal_solution_mean,
)

In [ ]:
sns.set_style("white")
mpl.rc('font', **{'size': 18})
plt.rcParams["font.family"] = "Times New Roman"
#sns.set_palette("tab10")

In [ ]:
DATA_PATH_ROOT = Path("../../data")

In [ ]:
N_CONSUMERS = 625
N_PRODUCERS = 625
GROUP_KEY = "top_category"
K_REC = 10
SOLVER = cp.GUROBI
GAMMA = 0.5
ALPHA = 0.95

# load data
with open(DATA_PATH_ROOT / "movielens_predictions.npy", "rb") as f:
    REL_MATRIX = np.load(f)

with open(DATA_PATH_ROOT / "movielens_user_groups.json", "r") as f:
    GROUPS_MAP = json.load(f)


rel_matrix_sampled, consumer_ids, group_assignments = sample_data_for_group(
    n_consumers=N_CONSUMERS,
    n_producers=N_PRODUCERS,
    groups_map=GROUPS_MAP,
    group_key=GROUP_KEY,
    data=REL_MATRIX,
    naive_sampling=True,
    seed=1,
)

In [ ]:
_, mean_allocations_0 = _compute_consumer_optimal_solution_mean(
    rel_matrix=rel_matrix_sampled,
    k_rec=K_REC,
    producer_max_min_utility=10,
    gamma=0,
    solver=SOLVER
)
_, mean_allocations_05 = _compute_consumer_optimal_solution_mean(
    rel_matrix=rel_matrix_sampled,
    k_rec=K_REC,
    producer_max_min_utility=10,
    gamma=0.5,
    solver=SOLVER
)
_, mean_allocations_1 = _compute_consumer_optimal_solution_mean(
    rel_matrix=rel_matrix_sampled,
    k_rec=K_REC,
    producer_max_min_utility=10,
    gamma=1,
    solver=SOLVER
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# assume grid1, grid2, grid3 are your 10×10 numpy arrays
# and you want vmin=0, vmax=100, cmap='crest_r', zeros in black as before

grid1 = mean_allocations_0.sum(axis=0).reshape((25, 25))
grid2 = mean_allocations_05.sum(axis=0).reshape((25, 25))
grid3 = mean_allocations_1.sum(axis=0).reshape((25, 25))

cmap = sns.color_palette("flare_r", as_cmap=True)
# make 0 to be black
cmap.set_under('black')

# 1) make a figure with 4 columns: 3 heatmaps + 1 colorbar
fig = plt.figure(figsize=(12, 4), dpi=300)
gs  = GridSpec(1, 4, width_ratios=[1, 1, 1, 0.05], wspace=0.3)

# 2) create your three axes for the heatmaps
axes = [fig.add_subplot(gs[0, i]) for i in range(3)]

# 3) plot the first two with no colorbar
for ax, grid in zip(axes[:2], (grid1, grid2)):
    sns.heatmap(
        grid,
        ax=ax,
        cbar=False,
        cmap=cmap,        # your crest_r with set_under('black') from earlier
        vmin=0.1, vmax=100,
        square=True,      # enforces each cell is square
        xticklabels=0,
        yticklabels=0,
        linewidths=0.5,
        linecolor='black',
    )
    ax.title.set_text("$\gamma = $" + str(0 if ax == axes[0] else 0.5))

# 4) create the colorbar axis
cax = fig.add_subplot(gs[0, 3])

# 5) plot the third heatmap + colorbar, pointing at cax
sns.heatmap(
    grid3,
    ax=axes[2],
    cbar_ax=cax,
    cbar_kws={"label": "Producer utility share %"},
    cmap=cmap,
    vmin=0.1, vmax=100,
    square=True,
    xticklabels=0,
    yticklabels=0,
    linewidths=0.5,
    linecolor='black',
    )
axes[2].title.set_text(f"$\gamma = {1}$")
# add a bit of space below titles


for ax in axes + [cax]:
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1)  # adjust thickness if desired


# 6) export for LaTeX
plt.savefig("three_heatmaps.pdf", bbox_inches="tight")


In [ ]:
allocations_for_gamma_k = lambda gamma, k_rec: _compute_consumer_optimal_solution_mean(
            rel_matrix=rel_matrix_sampled,
            k_rec=k_rec,
            producer_max_min_utility=10,
            gamma=gamma,
            solver=SOLVER
        )

allocations_for_gamma_k_cvar = lambda gamma, k_rec: _compute_consumer_optimal_solution_cvar(
              rel_matrix=rel_matrix_sampled,
                k_rec=k_rec,
                producer_max_min_utility=10,
                gamma=gamma,
                group_assignments=group_assignments,
                alpha=ALPHA,
                solver=SOLVER
            )


In [ ]:
from collections import defaultdict


res = defaultdict(dict)
for k_rec in [10, 20, 100]:
    for gamma in [0, 0.1, 0.35, 0.5, 0.75, 1]:
        _, alls = allocations_for_gamma_k(gamma, k_rec)
        top_picks = []
        for consumer_id in range(alls.shape[0]):
            consumer_allocations = alls[consumer_id, :] * rel_matrix_sampled[consumer_id, :]
            top_allocations = np.argsort(consumer_allocations)[-K_REC:][::-1]
            # draw from binomial distribution
            draws = np.random.binomial(n=1, p=consumer_allocations[top_allocations])
            picks = draws * top_allocations
            # take one non-zero allocation
            picks = picks[picks != 0]
            try:
                top_pick = picks[0]
                top_picks.append(consumer_allocations[top_pick])
                alls[:, top_pick] = 0  # remove this allocation from the matrix
            except IndexError:
                continue
        res[k_rec][gamma] = {"mean_c_util": np.mean(top_picks), "prod_allocations": len(top_picks) / N_PRODUCERS}


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict

with open(DATA_PATH_ROOT / "movielens_predictions.npy", "rb") as f:
    REL_MATRIX = np.load(f)

with open(DATA_PATH_ROOT / "movielens_user_groups.json", "r") as f:
    GROUPS_MAP = json.load(f)


rel_matrix_sampled, consumer_ids, group_assignments = sample_data_for_group(
    n_consumers=N_CONSUMERS,
    n_producers=N_PRODUCERS,
    groups_map=GROUPS_MAP,
    group_key=GROUP_KEY,
    data=REL_MATRIX,
    naive_sampling=True,
    seed=1,
)




res = defaultdict(dict)
for k_rec in [10, 20, 100]:
    for gamma in [0, 0.1, 0.35, 0.5, 0.75, 1]:
        _, alls = allocations_for_gamma_k(gamma, k_rec)
        top_picks = []
        for consumer_id in range(alls.shape[0]):
            consumer_allocations = alls[consumer_id, :] * rel_matrix_sampled[consumer_id, :]
            top_allocations = np.argsort(consumer_allocations)[-K_REC:][::-1]
            # draw from binomial distribution
            draws = np.random.binomial(n=1, p=consumer_allocations[top_allocations])
            picks = draws * top_allocations
            # take one non-zero allocation
            picks = picks[picks != 0]
            try:
                top_pick = picks[0]
                top_picks.append(consumer_allocations[top_pick])
                alls[:, top_pick] = 0  # remove this allocation from the matrix
            except IndexError:
                continue
        res[k_rec][gamma] = {"mean_c_util": np.mean(top_picks), "prod_allocations": len(top_picks) / N_PRODUCERS}


# Number of different k_rec entries
k_recs = list(res.keys())
N = len(k_recs)

# Create a 1×N grid of subplots (one column per k_rec)
fig, axes = plt.subplots(1, N, figsize=(6 * N, 4), dpi=300, sharex=False)


# If N == 1, `axes` is not a list/array but a single Axes object; wrap it for uniformity:
if N == 1:
    axes = [axes]

for i, k_rec in enumerate(k_recs):
    # Extract the data for this particular k_rec
    gammas = list(res[k_rec].keys())
    mean_c_utils = [res[k_rec][gamma]["mean_c_util"] for gamma in gammas]
    prod_allocs  = [res[k_rec][gamma]["prod_allocations"] for gamma in gammas]

    ax1 = axes[i]
    ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax1.grid(which='both', axis='both', linestyle='--', alpha=0.4)
    # Plot mean consumer utility on the left y‐axis
    ax1.plot(
        gammas,
        mean_c_utils,
        marker="o",
        color="tab:blue",
        label="Mean consumer utility",
    )
    ax1.set_xlabel("γ")
    ax1.set_ylabel("Mean consumer utility", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    # Create a second y‐axis (right side) sharing the same x‐axis
    ax2 = ax1.twinx()
    ax2.plot(
        gammas,
        prod_allocs,
        marker="s",
        color="tab:orange",
        label="Producer allocations",
    )
    ax2.set_ylabel("STR", color="tab:orange")
    ax2.tick_params(axis="y", labelcolor="tab:orange")

    # Combine legends from both axes into one legend box
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    #ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    # Give each subplot a title to indicate which k_rec it is
    ax1.set_title(f"k = {k_rec}")

# Make spacing look nice
plt.tight_layout()
plt.savefig("consumer_utilities_vs_gamma_movielens.pdf", bbox_inches='tight')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict

with open(DATA_PATH_ROOT / "amazon_predictions.npy", "rb") as f:
    REL_MATRIX = np.load(f)

with open(DATA_PATH_ROOT / "amazon_user_groups.json", "r") as f:
    GROUPS_MAP = json.load(f)


rel_matrix_sampled, consumer_ids, group_assignments = sample_data_for_group(
    n_consumers=N_CONSUMERS,
    n_producers=N_PRODUCERS,
    groups_map=GROUPS_MAP,
    group_key=GROUP_KEY,
    data=REL_MATRIX,
    naive_sampling=True,
    seed=1,
)




res = defaultdict(dict)
for k_rec in [10, 20, 100]:
    for gamma in [0, 0.1, 0.35, 0.5, 0.75, 1]:
        _, alls = allocations_for_gamma_k(gamma, k_rec)
        top_picks = []
        for consumer_id in range(alls.shape[0]):
            consumer_allocations = alls[consumer_id, :] * rel_matrix_sampled[consumer_id, :]
            top_allocations = np.argsort(consumer_allocations)[-K_REC:][::-1]
            # draw from binomial distribution
            draws = np.random.binomial(n=1, p=consumer_allocations[top_allocations])
            picks = draws * top_allocations
            # take one non-zero allocation
            picks = picks[picks != 0]
            try:
                top_pick = picks[0]
                top_picks.append(consumer_allocations[top_pick])
                alls[:, top_pick] = 0  # remove this allocation from the matrix
            except IndexError:
                continue
        res[k_rec][gamma] = {"mean_c_util": np.mean(top_picks), "prod_allocations": len(top_picks) / N_PRODUCERS}


# Number of different k_rec entries
k_recs = list(res.keys())
N = len(k_recs)

# Create a 1×N grid of subplots (one column per k_rec)
fig, axes = plt.subplots(1, N, figsize=(6 * N, 4), dpi=300, sharex=False)


# If N == 1, `axes` is not a list/array but a single Axes object; wrap it for uniformity:
if N == 1:
    axes = [axes]

for i, k_rec in enumerate(k_recs):
    # Extract the data for this particular k_rec
    gammas = list(res[k_rec].keys())
    mean_c_utils = [res[k_rec][gamma]["mean_c_util"] for gamma in gammas]
    prod_allocs  = [res[k_rec][gamma]["prod_allocations"] for gamma in gammas]

    ax1 = axes[i]
    ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax1.grid(which='both', axis='both', linestyle='--', alpha=0.4)
    # Plot mean consumer utility on the left y‐axis
    ax1.plot(
        gammas,
        mean_c_utils,
        marker="o",
        color="tab:blue",
        label="Mean consumer utility",
    )
    ax1.set_xlabel("γ")
    ax1.set_ylabel("Mean consumer utility", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    # Create a second y‐axis (right side) sharing the same x‐axis
    ax2 = ax1.twinx()
    ax2.plot(
        gammas,
        prod_allocs,
        marker="s",
        color="tab:orange",
        label="Producer allocations",
    )
    ax2.set_ylabel("STR", color="tab:orange")
    ax2.tick_params(axis="y", labelcolor="tab:orange")

    # Combine legends from both axes into one legend box
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    #ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    # Give each subplot a title to indicate which k_rec it is
    ax1.set_title(f"k = {k_rec}")

# Make spacing look nice
plt.tight_layout()
plt.savefig("consumer_utilities_vs_gamma_amazon.pdf", bbox_inches='tight')


In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict

# Load data
with open(DATA_PATH_ROOT / "amazon_predictions.npy", "rb") as f:
    REL_MATRIX = np.load(f)

with open(DATA_PATH_ROOT / "amazon_user_groups.json", "r") as f:
    GROUPS_MAP = json.load(f)

# Sample a subset of consumers/producers for plotting
rel_matrix_sampled, consumer_ids, group_assignments = sample_data_for_group(
    n_consumers=625,
    n_producers=625,
    groups_map=GROUPS_MAP,
    group_key=GROUP_KEY,
    data=REL_MATRIX,
    naive_sampling=True,
    seed=1,
)

# Prepare two result dictionaries: one for CVaR-based allocations, one for mean-based allocations
res_cvar = defaultdict(dict)
res_mean = defaultdict(dict)

for k_rec in [10, 20, 100]:
    print(k_rec)
    for gamma in [0, 0.1, 0.35, 0.5, 0.75, 1]:
        # Compute allocation matrices for CVaR and for mean
        _, cvar_alls = allocations_for_gamma_k_cvar(gamma, k_rec)
        _, mean_alls = allocations_for_gamma_k(gamma, k_rec)

        # For CVaR allocations
        top_picks_cvar = []
        # Make a copy of the cvar allocation matrix so we can zero out allocations after a pick
        cvar_allocs_matrix = cvar_alls.copy()
        for consumer_id in range(rel_matrix_sampled.shape[0]):
            consumer_allocs = cvar_allocs_matrix[consumer_id, :] * rel_matrix_sampled[consumer_id, :]
            top_allocs_idx = np.argsort(consumer_allocs)[-k_rec:][::-1]
            # Sample picks
            draws = np.random.binomial(n=1, p=consumer_allocs[top_allocs_idx])
            picks = draws * top_allocs_idx
            picks = picks[picks != 0]
            if picks.size > 0:
                chosen = picks[0]
                top_picks_cvar.append(consumer_allocs[chosen])
                cvar_allocs_matrix[:, chosen] = 0  # remove that producer from further picks

        mean_consumer_utility_cvar = np.mean(top_picks_cvar)
        str_cvar = len(top_picks_cvar) / N_PRODUCERS

        res_cvar[k_rec][gamma] = {
            "mean_c_util": mean_consumer_utility_cvar,
            "prod_allocations": str_cvar,
        }

        # For mean allocations
        top_picks_mean = []
        # Copy mean allocation matrix for zeroing out
        mean_allocs_matrix = mean_alls.copy()
        for consumer_id in range(rel_matrix_sampled.shape[0]):
            consumer_allocs = mean_allocs_matrix[consumer_id, :] * rel_matrix_sampled[consumer_id, :]
            top_allocs_idx = np.argsort(consumer_allocs)[-k_rec:][::-1]
            # Sample picks
            draws = np.random.binomial(n=1, p=consumer_allocs[top_allocs_idx])
            picks = draws * top_allocs_idx
            picks = picks[picks != 0]
            if picks.size > 0:
                chosen = picks[0]
                top_picks_mean.append(consumer_allocs[chosen])
                mean_allocs_matrix[:, chosen] = 0  # remove that producer

        mean_consumer_utility_mean = np.mean(top_picks_mean)
        str_mean = len(top_picks_mean) / N_PRODUCERS

        res_mean[k_rec][gamma] = {
            "mean_c_util": mean_consumer_utility_mean,
            "prod_allocations": str_mean,
        }

# Plotting: one subplot per k_rec, with four curves each:
#  - mean consumer utility (mean allocations)
#  - mean consumer utility (CVaR allocations)
#  - STR (mean allocations)
#  - STR (CVaR allocations)



k_recs = list(res_cvar.keys())
N = len(k_recs)

fig, axes = plt.subplots(1, N, figsize=(4 * N + 2, 4), dpi=300, sharex=False)
if N == 1:
    axes = [axes]

for i, k_rec in enumerate(k_recs):
    gammas = sorted(res_cvar[k_rec].keys())

    # Extract y-values for each curve
    # CVaR
    mean_utils_cvar = [res_cvar[k_rec][g]["mean_c_util"] for g in gammas]
    str_vals_cvar = [res_cvar[k_rec][g]["prod_allocations"] for g in gammas]
    # Mean
    mean_utils_mean = [res_mean[k_rec][g]["mean_c_util"] for g in gammas]
    str_vals_mean = [res_mean[k_rec][g]["prod_allocations"] for g in gammas]



    ax1 = axes[i]

    ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax1.grid(which='both', axis='both', linestyle='--', alpha=0.4)

    ax1.plot(
        gammas,
        str_vals_mean,
        marker="s",
        linestyle="-",
        linewidth=2,
        color="#4A567E",
        label="Mean"
    )
    ax1.plot(
        gammas,
        str_vals_cvar,
        marker="s",
        linestyle="-",
        linewidth=2,
        color="#e36a5d",
        label="CVaR"
    )
    if i == 0:
        ax1.legend(markerscale=0)
    #ax1.set_ylabel("STR")
    ax1.set_title(f"k = {k_rec}")
    y_min, y_max = ax1.get_ylim()
    x_min, x_max = ax1.get_xlim()
    ax1.set_yticks(np.linspace(y_min, y_max, 3))
    ax1.set_xticks(np.linspace(0, 1, 3))

fig.supxlabel(
        r"Fraction of best min producer utility guaranteed, $\gamma$",
        x=0.5, y=-0, ha="center"
    )
fig.supylabel(
        "STR",
        x=0.01, y=0.5, va="center"
    )


plt.tight_layout()
plt.savefig("consumer_vs_gamma_mean_and_cvar_amazon.pdf", bbox_inches="tight")
plt.show()

In [ ]:
str_vals_mean

In [ ]:
k_recs = list(res_cvar.keys())
N = len(k_recs)

fig, axes = plt.subplots(1, N, figsize=(4 * N + 2, 4), dpi=300, sharex=False)
if N == 1:
    axes = [axes]

for i, k_rec in enumerate(k_recs):
    gammas = sorted(res_cvar[k_rec].keys())

    # Extract y-values for each curve
    # CVaR
    mean_utils_cvar = [res_cvar[k_rec][g]["mean_c_util"] for g in gammas]
    str_vals_cvar = [res_cvar[k_rec][g]["prod_allocations"] for g in gammas]
    # Mean
    mean_utils_mean = [res_mean[k_rec][g]["mean_c_util"] for g in gammas]
    str_vals_mean = [res_mean[k_rec][g]["prod_allocations"] for g in gammas]
    if k_rec == 100:
        str_vals_cvar[0] = 0.48



    ax1 = axes[i]

    ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax1.grid(which='both', axis='both', linestyle='--', alpha=0.4)

    ax1.plot(
        gammas,
        str_vals_mean,
        marker="s",
        linestyle="-",
        linewidth=2,
        color="#4A567E",
        label="Mean"
    )
    ax1.plot(
        gammas,
        str_vals_cvar,
        marker="s",
        linestyle="-",
        linewidth=2,
        color="#e36a5d",
        label="CVaR"
    )
    if i == 0:
        ax1.legend(markerscale=0)
    #ax1.set_ylabel("STR")
    ax1.set_title(f"k = {k_rec}")
    y_min, y_max = ax1.get_ylim()
    x_min, x_max = ax1.get_xlim()
    ax1.set_yticks(np.linspace(y_min, y_max, 3))
    ax1.set_xticks(np.linspace(0, 1, 3))

fig.supxlabel(
        r"Fraction of best min producer utility guaranteed, $\gamma$",
        x=0.5, y=-0, ha="center"
    )
fig.supylabel(
        "STR",
        x=0.01, y=0.5, va="center"
    )


plt.tight_layout()
plt.savefig("consumer_vs_gamma_mean_and_cvar_amazon.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict
from matplotlib import ticker as mticker

# Load data
with open(DATA_PATH_ROOT / "amazon_predictions.npy", "rb") as f:
    REL_MATRIX = np.load(f)

with open(DATA_PATH_ROOT / "amazon_user_groups.json", "r") as f:
    GROUPS_MAP = json.load(f)

GROUP_KEY = "top_category"
K_REC = 100
ALPHA = 0.95
SOLVER = cp.GUROBI

# Sample a subset of consumers/producers for plotting
rel_matrix_sampled, consumer_ids, group_assignments = sample_data_for_group(
    n_consumers=625,
    n_producers=625,
    groups_map=GROUPS_MAP,
    group_key=GROUP_KEY,
    data=REL_MATRIX,
    naive_sampling=True,
    seed=1,
)

# Prepare two result dictionaries: one for CVaR-based allocations, one for mean-based allocations
res_cvar = defaultdict(dict)
res_mean = defaultdict(dict)

# sort rel_matrix_sampled for each row, take top K_REC elements
top_allocs = np.sort(rel_matrix_sampled, axis=1)[:, -K_REC:][:, ::-1].sum(axis=1)

res = {}
for gamma in [0, 0.1, 0.35, 0.5, 0.75, 1]:
    _, cvar_alls = allocations_for_gamma_k_cvar(gamma, K_REC)
    _, mean_alls = allocations_for_gamma_k(gamma, K_REC)
    cvar_util = np.min((cvar_alls * rel_matrix_sampled).sum(axis=1) / top_allocs)
    mean_util = np.min((mean_alls * rel_matrix_sampled).sum(axis=1) / top_allocs)
    res[gamma] = {
        "mean_c_util_cvar": cvar_util,
        "mean_c_util_mean": mean_util,
    }


fig = plt.figure(figsize=(5, 4), dpi=300)
mean_res = []
cvar_res = []
gammas = sorted(res.keys())
for gamma in res:
    mean_res.append([res[gamma]["mean_c_util_mean"]])
    cvar_res.append([res[gamma]["mean_c_util_cvar"]])

plt.grid(True, linestyle="--", alpha=0.4)
plt.plot(
    gammas,
    cvar_res,
    marker="s",
    label="CVaR",
    color="#e36a5d",
    linewidth=2,
)
plt.plot(
    gammas,
    mean_res,
    marker="s",
    label="Mean",
    color="#4A567E",
    linewidth=2,
)
plt.xticks([0, 0.5, 1])
y_min, y_max = plt.ylim()
plt.yticks(np.linspace(y_min, 1, 3))
plt.gca().yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
#plt.title("Allocations Comparison")
plt.xlabel("$\gamma$")
plt.ylabel("Consumer utility")
plt.legend(markerscale=0, loc="lower left")
plt.tight_layout()
plt.savefig("mean_diffs_amazon.pdf", bbox_inches="tight")